In [ ]:
import asyncio
import aiohttp
import json

async def debug_sse(url):
    print(f"--- Debugging SSE: {url} ---")
    async with aiohttp.ClientSession() as session:
        try:
            async with session.get(f"{url}/sse") as resp:
                print(f"Status: {resp.status}")
                if resp.status != 200:
                    print(f"Error Body: {await resp.text()}")
                    return
                
                # Read first line to get endpoint
                line = await resp.content.readline()
                line_str = line.decode().strip()
                print(f"First line: {line_str}")
                
                if not line_str.startswith("event: endpoint"):
                    print("Unexpected first event")
                    return
                
                next_line = await resp.content.readline()
                next_line_str = next_line.decode().strip()
                print(f"Second line: {next_line_str}")
                
                if not next_line_str.startswith("data: "):
                    print("Unexpected second line")
                    return
                
                msg_endpoint = next_line_str.replace("data: ", "").strip()
                print(f"Message Endpoint: {msg_endpoint}")
                
                # Now try to POST listTools
                post_url = f"{url}{msg_endpoint}"
                print(f"Posting listTools to: {post_url}")
                
                payload = {
                    "jsonrpc": "2.0",
                    "id": 1,
                    "method": "listTools"
                }
                
                async with session.post(post_url, json=payload) as post_resp:
                    print(f"POST Status: {post_resp.status}")
                    print(f"POST Response: {await post_resp.text()}")

        except Exception as e:
            print(f"Exception: {e}")

# Run for ServiceNow
await debug_sse("https://servicenow-mcp-254356041555.us-central1.run.app")
